In [1]:
import pandas as pd

# 포인트 사용 기록 테이블

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_pointhistory`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

          id  delta_point                created_at  user_id  \
0  145379907         -500 2023-05-18 12:33:59+00:00  1159163   
1  145400847         -500 2023-05-18 12:35:09+00:00  1243371   
2  145467849         -500 2023-05-18 12:38:46+00:00  1295255   
3  145644563         -500 2023-05-18 12:48:18+00:00   876611   
4  145705235        -1000 2023-05-18 12:51:30+00:00  1117193   

   user_question_record_id  
0                 71772936  
1                 40846522  
2                 69224499  
3                 71863782  
4                 54641410  


## 결측치 확인 및 데이터 정보 확인

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2338918 entries, 0 to 2338917
Data columns (total 5 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   id                       Int64              
 1   delta_point              Int64              
 2   created_at               datetime64[us, UTC]
 3   user_id                  Int64              
 4   user_question_record_id  Int64              
dtypes: Int64(4), datetime64[us, UTC](1)
memory usage: 98.1 MB


In [5]:
df.isna().sum()

id                            0
delta_point                   0
created_at                    0
user_id                       0
user_question_record_id    2992
dtype: int64

* `user_question_record_id` 2992건 결측 발견

In [6]:
df[df['user_question_record_id'].isna()]

,id,delta_point,created_at,user_id,user_question_record_id
1691,330788563,200,2023-06-22 11:55:43+00:00,1120688,<NA>
1701,331514240,500,2023-06-24 11:03:13+00:00,1213990,<NA>
1705,331693207,200,2023-06-24 16:07:28+00:00,1402393,<NA>
1706,331694812,500,2023-06-24 16:12:04+00:00,1402393,<NA>
1708,331844825,200,2023-06-25 06:26:29+00:00,1263081,<NA>
...,...,...,...,...,...
1188054,340566243,80,2024-02-02 19:15:35+00:00,875012,<NA>
1188056,340599302,777,2024-02-27 14:41:47+00:00,1318890,<NA>
1188058,340614885,50,2024-03-17 02:09:08+00:00,1525512,<NA>
1188059,340616930,50,2024-03-19 02:37:26+00:00,1227249,<NA>


In [7]:
# 1. 각각의 집계 결과 계산
na_size = df[df['user_question_record_id'].isna()].groupby('delta_point').size()
total_size = df.groupby('delta_point').size()

# 2. 하나의 데이터프레임으로 나란히 병합
result = pd.concat([na_size, total_size], axis=1, keys=['na_count', 'total_count']).fillna(0)

# 3. 비율(%) 파생변수까지 추가하면 더 직관적입니다
result['na_ratio(%)'] = (result['na_count'] / result['total_count'] * 100).round(2)

result

,na_count,total_count,na_ratio(%)
delta_point,,,
-30,1.0,1,100.0
50,322.0,322,100.0
60,47.0,47,100.0
70,16.0,16,100.0
80,10.0,10,100.0
90,7.0,7,100.0
100,10.0,10,100.0
110,6.0,6,100.0
120,6.0,6,100.0


In [8]:
# 전처리 수행 대상 테이블 호출
temp_sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.events`
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head())

   id           title  plus_point event_type  is_expired  \
0   1   코드잇 은행 가입 이벤트         500       FCFS           1   
1   3   예고 영상 기대평 이벤트         500       FCFS           1   
2   2  코드잇 멤버십 가입 이벤트        1000       FCFS           1   

                 created_at  
0 2023-06-20 11:56:38+00:00  
1 2023-09-24 17:05:59+00:00  
2 2023-08-08 07:43:45+00:00  


* 200, 777, 1000에서 발견된 결측은 충전 과정에서 발생된 history로 판단된다.
* 또한 500포인트와 1000포인트를 지급하는 이벤트를 진행했으므로, 500포인트 역시 user_question_record_id 결측 설명이 가능.

* -30, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 210, 220, 230, 240, 250, 260, 270, 280, 300 포인트 확인 필요

* 과거 다른 형태의 요금제가 있었는지 확인

In [9]:
# 전처리 수행 대상 테이블 호출
temp_sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_paymenthistory`
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

print(temp_df.head())

      id  productId phone_type                created_at  user_id
0  89584  heart.777          A 2023-06-06 04:58:49+00:00   835888
1  89585  heart.200          A 2023-06-06 04:59:22+00:00   835888
2   2403  heart.777          A 2023-05-14 04:22:44+00:00   837641
3  79774  heart.777          A 2023-05-29 10:13:55+00:00   837737
4    195  heart.777          A 2023-05-13 23:10:10+00:00   837842


In [10]:
temp_df.groupby(by='productId').count()

,id,phone_type,created_at,user_id
productId,,,,
heart.1000,19309,19309,19309,19309
heart.200,15822,15822,15822,15822
heart.4000,2136,2136,2136,2136
heart.777,57873,57873,57873,57873


In [11]:
print(df['created_at'].min())
print(df['created_at'].max())
print(temp_df['created_at'].min())
print(temp_df['created_at'].max())

2023-04-28 12:27:49+00:00
2024-05-08 01:36:18+00:00
2023-05-13 21:28:34+00:00
2024-05-08 14:12:45+00:00


* 요금제의 변경은 없었던 것으로 판단되며,
* 기간 중 4000포인트의 충전이 꽤 있었으나, 포인트 기록 히스토리에서는 보이지 않고,
* 777포인트도 수의 차이가 보임.

* 기획서 상 포인트 수급처를 추가로 확인해 본 결과 출석체크도 있긴 함.
* 그리고 연속 출석 일 수등 다양한 조건에 따라 보너스 포인트를 주기도 함. (기획서 상 240포인트는 확인)
* (추정) 이번주 출석 일 수 / 이번달 출석 일 수 / 연속 출석 일 수 등 다양한 조합에 따라 포인트가 쌓이는 것이 아닐지 추측
* 또한 주요 포인트 사용처는 초성확인으로 각각 200포인트 , 500포인트, 1000포인트가 소모.
* -10 포인트는 받은 핑에 대한 답장 시 소모
* -300, -30 등에 대한 사용처 추가 확인이 필요해 보임.

In [12]:
# -30 포인트 확인
df[df['delta_point'] == -30]

,id,delta_point,created_at,user_id,user_question_record_id
22934,338885967,-30,2023-09-02 11:51:11+00:00,1250603,<NA>


In [13]:
temp_sql = f"""
    SELECT * FROM {PROJECT_ID}.{DATA_SET}.accounts_user
    WHERE id = 1250603;
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,id,is_superuser,is_staff,gender,point,friend_id_list,is_push_on,created_at,block_user_id_list,hide_user_id_list,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
0,1250603,0,0,F,25,"[1185408, 1206529, 1377288, 1211020, 1259413, ...",1,2023-05-14 06:10:22.407500+00:00,[],[],N,0,0,0,29,49478


In [14]:
temp_sql = f"""
    SELECT * FROM {PROJECT_ID}.{DATA_SET}.accounts_group
    WHERE id = 49478;
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,id,grade,class_num,school_id
0,49478,1,2,1719


In [15]:
temp_sql = f"""
    SELECT * FROM {PROJECT_ID}.{DATA_SET}.accounts_school
    WHERE id = 1719;
"""

# 판다스 데이터프레임으로 변환
temp_df = client.query(temp_sql).to_dataframe()

temp_df.head()

,id,address,student_count,school_type
0,1719,울산광역시 울주군,550,H


In [16]:
df[df['user_id'] == 1250603].sort_values('created_at')

,id,delta_point,created_at,user_id,user_question_record_id
911957,84665376,13,2023-05-14 06:20:47+00:00,1250603,42226336
118028,84666936,5,2023-05-14 06:20:53+00:00,1250603,42227088
1445784,84775674,7,2023-05-14 06:28:05+00:00,1250603,42280045
2153135,85387563,14,2023-05-14 07:08:41+00:00,1250603,42577955
1595112,85395447,8,2023-05-14 07:09:13+00:00,1250603,42581789
...,...,...,...,...,...
759750,337622919,11,2023-08-04 01:29:42+00:00,1250603,160353240
343984,337622925,7,2023-08-04 01:29:49+00:00,1250603,160353243
871229,337622936,12,2023-08-04 01:29:59+00:00,1250603,160353248
22934,338885967,-30,2023-09-02 11:51:11+00:00,1250603,<NA>


* 해당 기록이 찍힌 유저 자체는 특별할 것이 없어보이지만, -30포인트가 발생한 사례는 단 한 건으로 오류로 보입니다.
* 업체 측에서 오류에 대한 보상을 확인해보려고 했으나, 관련된 움직임은 보여지지 않았음.

In [17]:
user_df = df[df['user_id'] == 1250603]

user_df[
    user_df[['delta_point', 'created_at', 'user_id']].duplicated(keep=False)
].sort_values('created_at')

,id,delta_point,created_at,user_id,user_question_record_id
2085103,183037929,13,2023-05-21 10:05:08+00:00,1250603,88981448
2134286,183037932,13,2023-05-21 10:05:08+00:00,1250603,88981448
621200,183056497,10,2023-05-21 10:06:26+00:00,1250603,88990024
645113,183056490,10,2023-05-21 10:06:26+00:00,1250603,88990024


* 또 다른 가능성으로 포인트 중복 수령 발생으로 인해 회수된 포인트가 아닐지? 확인 진행.
* 의심이 되는 가능성이지만, 데이터 상 명료하게 보이지는 않음.
* 해당 데이터가 단 한건이고, 보유 포인트의 영향을 크게 주지 않는다고 판단하여, 삭제를 진행함.

In [18]:
# -300 포인트 확인
df[df['delta_point'] == -300].sort_values('created_at')

,id,delta_point,created_at,user_id,user_question_record_id
23113,808783,-300,2023-04-28 14:31:21+00:00,849479,773997
2650,808861,-300,2023-04-28 14:31:49+00:00,849452,788254
2651,810295,-300,2023-04-28 14:41:47+00:00,849762,787307
2652,812641,-300,2023-04-28 14:55:08+00:00,849670,782368
1160974,815628,-300,2023-04-28 15:15:43+00:00,849439,790323
...,...,...,...,...,...
1214594,145949736,-300,2023-05-18 13:03:24+00:00,1303729,56455390
45547,145960673,-300,2023-05-18 13:03:59+00:00,1366896,71751550
12659,145995077,-300,2023-05-18 13:05:49+00:00,1370566,72053972
1181668,146077050,-300,2023-05-18 13:09:45+00:00,1201068,45569578


In [19]:
# 각 사용 금액 날짜 기록 범주 확인
print('10포인트')
display(df[df['delta_point'] == -10]['created_at'].min())
display(df[df['delta_point'] == -10]['created_at'].max())
print('200포인트')
display(df[df['delta_point'] == -200]['created_at'].min())
display(df[df['delta_point'] == -200]['created_at'].max())
print('300포인트')
display(df[df['delta_point'] == -300]['created_at'].min())
display(df[df['delta_point'] == -300]['created_at'].max())
print('500포인트')
display(df[df['delta_point'] == -500]['created_at'].min())
display(df[df['delta_point'] == -500]['created_at'].max())
print('1000포인트')
display(df[df['delta_point'] == -1000]['created_at'].min())
display(df[df['delta_point'] == -1000]['created_at'].max())

10포인트


Timestamp('2023-05-18 12:18:54+0000', tz='UTC')

Timestamp('2023-06-07 18:02:41+0000', tz='UTC')

200포인트


Timestamp('2023-05-18 12:19:05+0000', tz='UTC')

...

Timestamp('2023-05-18 12:24:21+0000', tz='UTC')

Timestamp('2024-05-07 02:26:58+0000', tz='UTC')

1000포인트


Timestamp('2023-05-18 12:51:30+0000', tz='UTC')

Timestamp('2024-04-30 10:33:13+0000', tz='UTC')

In [20]:
df[df['delta_point'] == -300].sort_values('created_at')

# 1. delta_point가 -300인 데이터만 먼저 필터링
filtered_df = df[df['delta_point'] == -300]

# 2. 유저ID와 질문기록ID가 중복된 행 추출 (sort_values 후 duplicated 적용)
target_cols = ['user_id', 'user_question_record_id']
result = filtered_df[filtered_df.duplicated(subset=target_cols, keep=False)].sort_values('created_at')

result

,id,delta_point,created_at,user_id,user_question_record_id
1188067,825616,-300,2023-04-28 16:39:11+00:00,850098,789980
1188080,864153,-300,2023-04-29 04:20:38+00:00,849559,827821
1188093,890018,-300,2023-04-29 07:28:19+00:00,850229,868172
1188118,922755,-300,2023-04-29 10:09:43+00:00,849535,897647
2714,966629,-300,2023-04-29 13:10:28+00:00,849634,886343
...,...,...,...,...,...
1181661,145771435,-300,2023-05-18 12:54:54+00:00,1017960,51714208
45545,145820697,-300,2023-05-18 12:57:19+00:00,1017960,51714208
45546,145896999,-300,2023-05-18 13:00:57+00:00,1253264,65297782
12658,145899373,-300,2023-05-18 13:01:03+00:00,1253264,65297782


* 소모 내역 변화 추이: 데이터 기록상 10, 200, 500, 1,000포인트 소모는 2023년 5월 18일부터 시작되었으며, 기존의 300포인트 소모 기록은 같은 날인 5월 18일을 기점으로 끊긴 것을 확인했습니다.

* 대규모 재화 업데이트 추정: 업데이트 반영 시간에 따른 약간의 오차를 감안하더라도, 5월 18일을 기점으로 플랫폼 내 재화 소모 체계에 대규모 업데이트가 진행된 것으로 추정됩니다.

* 300포인트의 기존 소모처 분석: 기획서상 관련 언급은 없으나, 질문 기록 ID와의 연관성을 볼 때 300포인트는 5월 18일 재화 업데이트 이전의 '초성 확인 기능' 소모 비용으로 판단됩니다.

* 단위 금액 추정: 동일 유저가 동일 레코드에 연속으로 재화를 소모한 기록이 존재하는 것으로 보아, 전체 초성이 아닌 '한 글자 초성 확인'에 대한 소모 금액으로 추정됩니다.

따라서 300포인트 소모 기록은 정상 데이터로 판단되며, 향후 재화 소모 관련 데이터 분석 시 해당 변경 이력을 유의하여 진행해 주시면 감사하겠습니다.

## 중복값 확인

In [21]:
df.duplicated().sum()

np.int64(0)

In [22]:
df[['delta_point', 'created_at', 'user_id', 'user_question_record_id']].duplicated().sum()

np.int64(1939)

* id값 제외시 중복 1939건 발견
* 중복 이벤트 가능성 있음 확인 필요.

In [23]:
# 중복 검사 대상 컬럼 목록
cols = ['delta_point', 'created_at', 'user_id', 'user_question_record_id']

# 1. 중복 데이터만 필터링한 뒤 해당 컬럼 기준으로 정렬해서 출력
duplicated_df = df[df[cols].duplicated(keep=False)].sort_values(by=cols)

# 2. 상위 10개 출력 (연달아 붙어있는 중복 데이터 확인)
with pd.option_context('display.max_columns', None):
    display(duplicated_df.head(10)) 

,id,delta_point,created_at,user_id,user_question_record_id
1244407,77956321,5,2023-05-13 16:26:32+00:00,1198097,38953394
1244408,77956327,5,2023-05-13 16:26:32+00:00,1198097,38953394
75839,80320871,5,2023-05-14 01:24:22+00:00,1234279,40107388
1244840,80320874,5,2023-05-14 01:24:22+00:00,1234279,40107388
76536,84225200,5,2023-05-14 05:52:04+00:00,1238029,42011932
1278631,84225197,5,2023-05-14 05:52:04+00:00,1238029,42011932
118111,85089876,5,2023-05-14 06:49:10+00:00,1091180,42433065
118112,85089882,5,2023-05-14 06:49:10+00:00,1091180,42433065
77426,88513465,5,2023-05-14 10:32:05+00:00,1091180,44098497
118763,88513469,5,2023-05-14 10:32:05+00:00,1091180,44098497


* 중복 발생한 데이터로 보는 것이 적합하다고 판단.

### 최종 처리 내용 정리

* -30 포인트 데이터 삭제
* 포인트 중복 수급 삭제

In [24]:
# 중복 데이터 1939건 삭제 진행 + -30포인트 삭제

delete_sql = f"""
DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_pointhistory`
WHERE id IN (
    SELECT id
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_pointhistory`
    QUALIFY ROW_NUMBER() OVER(PARTITION BY delta_point, user_id, created_at, user_question_record_id ORDER BY id ASC) > 1
) OR delta_point = -30;
"""

query_job = client.query(delete_sql)
query_job.result()  # DML 완료 대기

print(f"\n삭제 완료! 총 삭제된 행 수: {query_job.num_dml_affected_rows}건")


삭제 완료! 총 삭제된 행 수: 1940건
